# 松尾研 Physical AI 応用編1｜最終課題
## りんご粗選果ロボット — 実験再現ノートブック

**提出用資料更新：2026/11/01 04:54:20 JST**  
**実行対象：P104（物理動作10ケース）／P105（把持アップ・3D追従カメラ）**

### 研究目的
収穫現場での粗選果を想定し、りんごの外観画像に対するAIの判定を、3本指ロボットの**把持・等級別搬送・満杯時の停止・模擬回収後の再開・異物拒否**へ連動させる。シミュレーターにはPyBulletを使用した。

### 再現対象と検証実績
- **P104**：既知画像を含む10ケース（りんご9件＋非りんご1件）の物理実行。9件の等級・配置先が期待値と一致し、テニスボール1件は把持前に拒否した。
- **P105**：同じ連続実行に対して把持アップ・スライダー追従の2視点を追加撮影。両視点各656フレーム、追従位置のX方向変化は約1.39m。
- **工程107の独立したColab実行**：2026年11月1日、実行者提供の判定JSONと動画を照合し、`PHYSICS_10_OF_10 True`、`P107_NEW_RUN_RESULT PASS`、動画3本の検証を確認した。

これらは**開発用の既知画像を含む条件での再現実績**であり、未知のりんごに対する分類精度、実機での安全性、全ケース3秒以内の処理を保証しない。

### 実行方法（Google Colab）
1. すでに公開済みのGitHubからノートブックを開いた場合は、最初のセルの `REPO_URL` に**リポジトリのURL**を入力する。直接ZIPを使用する場合は空欄にし、`g108_public.zip` をアップロードする。
2. Colabの［ランタイム］→［すべてのセルを実行］を選ぶ。処理は**実験原本・ハッシュ照合 → 環境構築 → 物理実行と撮影 → 合否検査**の順に進む。
3. `P107_NEW_RUN_RESULT PASS` が表示され、今回生成された `P105_CAPTURE_VERDICT.json` と `P105_CAMERA_VIEWS.zip` を取得できれば合格である。

**再実行上の注意**：新規の作業フォルダに1回だけ実行する。途中で停止した場合はログを保存し、同じ作業フォルダに上書き・自動再試行しない。

### 提出動画の合成について
物理実行後に、`submission_video/` にある**工程98R2の承認済み背景画像**と実録3視点を組み合わせ、提出用の見せ方を再現できる。これは**物理実験の入力ではなく動画編集工程**であり、`submission_video/README.md` にコマンドを記載した。背景の装置は完成予想図であり、実機写真ではない。

> ノートブック出力に残る `P107_...` の名称は検証時の識別子である。提出版のファイル名と異なっても、実行アルゴリズムを示すバージョン違いではない。


In [ ]:
# 1/5｜実験リポジトリの取得
# 再現元の取得方法は二通り：Public GitHubのURL、または配布ZIPのアップロード。
# GitHubから取得する場合だけ REPO_URL を設定する。ZIPを使う場合は空欄のままにする。
REPO_URL = ""  # @param {type:"string"}

from pathlib import Path
import subprocess, sys, zipfile, re, hashlib
from google.colab import files

repo = Path('/content/P107_REPO')
if repo.exists():
    raise RuntimeError('STOP: P107_REPO_ALREADY_EXISTS; 既存の実行成果を消さず内容を確認する')

if REPO_URL.strip():
    url = REPO_URL.strip().rstrip('/')
    if not re.fullmatch(r'https://github\.com/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+(?:\.git)?', url):
        raise ValueError('GitHub PublicリポジトリのHTTPS URLを指定すること')
    subprocess.run(['git','clone','--depth','1',url,str(repo)],check=True)
    print('P107_SOURCE_GITHUB_CLONED', url)
else:
    print('配布用g108系ZIPを選択（g108_public.zipなど）')
    # Colabでは同名の再アップロード時に (1)、(2) が付くことがある。
    # 既存ZIPを検出した場合は再アップロードを要求しない。
    import io
    def g108_name_ok(name):
        return re.fullmatch(r'g108(?:_ready|_fix|_public)?(?: \(\d+\))?\.zip', name) is not None

    existing = sorted(
        (p for p in Path('/content').glob('g108*.zip') if g108_name_ok(p.name)),
        key=lambda p: p.stat().st_mtime, reverse=True
    )
    if existing:
        name = existing[0].name
        data = existing[0].read_bytes()
        print('P108_REUSE_UPLOADED_ZIP', name, len(data))
    else:
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise RuntimeError('STOP: ZIP_ONE_FILE_REQUIRED')
        name, data = next(iter(uploaded.items()))
        if not g108_name_ok(name):
            raise RuntimeError('STOP: g108系の配布ZIPを選択すること')
    with zipfile.ZipFile(io.BytesIO(data)) as z:
        if z.testzip() is not None:
            raise RuntimeError('STOP: ZIP_CRC_FAILURE')
        file_set = set(z.namelist())
        expected = {
            'P106_ASSET_MANIFEST.json', 'reproduction/p105_camera.py',
            'submission_video/p98r2_background.png',
            'submission_video/p105_compose.py', 'RUN_ALL_COLAB.ipynb'
        }
        missing = sorted(expected - file_set)
        if missing:
            raise RuntimeError('STOP: WRONG_ARCHIVE_MISSING ' + str(missing))
        for member in z.infolist():
            rel = Path(member.filename)
            if rel.is_absolute() or '..' in rel.parts:
                raise RuntimeError('STOP: UNSAFE_ZIP_ENTRY')
        repo.mkdir(parents=True, exist_ok=False)
        z.extractall(repo)
    print('P107_SOURCE_ZIP_EXTRACTED',len(data),'bytes')

required=['P106_ASSET_MANIFEST.json','reproduction/p105_camera.py',
          'reproduction/assemble_assets.py','verify_evidence.py',
          'verify_stage107.py','RUN_ALL_COLAB.ipynb']
missing=[x for x in required if not (repo/x).is_file()]
if missing:raise RuntimeError('STOP: REQUIRED_FILES_MISSING '+str(missing))
print('P107_REPOSITORY_READY',repo)


In [ ]:
# 2/5｜原本素材と保存済み監査記録の整合性確認
# 元写真・UV・テクスチャの復元とSHA-256照合を行う。
# これは保存済み資料の静的検査であり、新たなPyBullet実行ではない。
import subprocess, sys

def checked(*args):
    cmd=[sys.executable,*args]
    print('CHECK', ' '.join(str(x) for x in cmd),flush=True)
    subprocess.run(cmd,cwd=str(repo),check=True)

checked('verify_stage107.py')
checked('reproduction/assemble_assets.py')
checked('verify_assets.py')
checked('verify_evidence.py')
asset=repo/'assets/p92c_verified.zip'
assert asset.is_file() and asset.stat().st_size==50324332
print('P107_STATIC_AND_INPUT_VERIFIED')


In [ ]:
# 3/5｜Google Colabの実験環境の準備
# 必要な日本語フォント・FFmpeg・Python依存関係をインストールする。
# AI判定器やロボットの物理制御コードは変更しない。
import subprocess, sys
from pathlib import Path
font=Path('/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc')
if not font.is_file():
    subprocess.run(['apt-get','update','-qq'],check=True)
    subprocess.run(['apt-get','install','-y','-qq','fonts-noto-cjk'],check=True)
if not Path('/usr/bin/ffmpeg').is_file():
    subprocess.run(['apt-get','update','-qq'],check=True)
    subprocess.run(['apt-get','install','-y','-qq','ffmpeg'],check=True)
print('P107_DEPENDENCY_INSTALL_START',flush=True)
subprocess.run([sys.executable,'-m','pip','install','-r',str(repo/'requirements.txt')],check=True)
subprocess.run([sys.executable,'-c',
  'import pybullet, sklearn, trimesh, numpy, cv2, imageio; assert sklearn.__version__=="1.8.0"; print("P107_DEPENDENCY_PROBE_PASS")'],check=True)
print('P107_RUNTIME_READY',flush=True)


In [ ]:
# 4/5｜10ケースの物理試行と2視点カメラ撮影
# P105ランナーの一度の実行でP104相当の10ケースと追加撮影を行う。
# 途中停止時はログを保全し、同じ作業フォルダへの再実行を禁止する。
from pathlib import Path
import subprocess, sys
work=Path('/content/P107_CAPTURE_ONCE')
if work.exists():
    raise RuntimeError('STOP: CAPTURE_WORKSPACE_EXISTS; 過去ログを確認し、再実行しない')
log=Path('/content/P107_CAPTURE_ONCE.log')
if log.exists():
    raise RuntimeError('STOP: CAPTURE_LOG_EXISTS; 二重起動をしない')

cmd=[sys.executable,'-u',str(repo/'reproduction/p105_camera.py'),
     '--p92c',str(repo/'assets/p92c_verified.zip'),
     '--work',str(work)]
print('P107_REAL_PYBULLET_RUN_ONCE', ' '.join(cmd),flush=True)
with log.open('x',encoding='utf8') as out:
    proc=subprocess.Popen(cmd,cwd=str(repo),stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT,text=True,errors='replace',bufsize=1)
    line_count=0
    for line in proc.stdout:
        out.write(line)
        out.flush()
        line_count+=1
        if line_count%60==0 or any(x in line for x in (
            'P105_','P104_','RUN_PHYSICS','PASS','FAIL','STOP:',
            'RUNNER_RETURN_CODE','Traceback','Error','FRESH_AI_MODELS')):
            print(line.rstrip()[:1000],flush=True)
    rc=proc.wait()
print('P107_P105_PROCESS_RETURN_CODE',rc,'LOG',log,flush=True)
if rc!=0:
    raise RuntimeError('物理実行が正常終了していない。ログを保全し、自動再実行しない。')


In [ ]:
# 5/5｜今回の新規実行結果の合否判定
# 判定JSON・2カメラのフレーム数・カメラ移動・ZIP整合性を検証する。
# 同梱済みの過去の成功記録だけを根拠にPASSを出すことはない。
import json,zipfile
from pathlib import Path
from IPython.display import display, FileLink

v=work/'P105_CAPTURE_VERDICT.json'
z=work/'P105_CAMERA_VIEWS.zip'
if not v.is_file():raise RuntimeError('STOP: 新規実行判定ファイルがない')
r=json.loads(v.read_text(encoding='utf8'))
c=r.get('extra_cameras',{})
counts=c.get('camera_frame_counts',{})
passed=(r.get('rerun_physics_10of10') is True and
        r.get('reproduction_claim') is True and
        r.get('physics_exit_code') == 0 and
        r.get('python_changed_during_run') is False and
        c.get('real_pybullet_getCameraImage') is True and
        counts.get('grip')==counts.get('follow')==c.get('total_original_frames') and
        counts.get('grip')==656 and
        c.get('palm_x_motion_range_m',0)>.08 and
        not c.get('errors') and z.is_file())
if z.is_file():
    with zipfile.ZipFile(z) as archive:
        passed=passed and archive.testzip() is None
print('PHYSICS_10_OF_10',r.get('rerun_physics_10of10'))
print('CAMERA_FRAMES',counts)
print('CAMERA_X_TRAVEL_M',c.get('palm_x_motion_range_m'))
print('P107_NEW_RUN_RESULT','PASS' if passed else 'NOT_VERIFIED')
if not passed:
    raise RuntimeError('STOP: 新規試行は未合格。既存の証拠を成功扱いに置き換えない。')
display(FileLink(str(z)))
display(FileLink(str(v)))
print('完成した動画と監査結果のリンクを開いて取得できる')


## 実行結果の読み方と再現性の限界

- `P107_NEW_RUN_RESULT PASS` は**このランタイムで今回新たに実行した**物理試行およびカメラ監査が合格した場合のみ表示される。同梱の `evidence/` にある過去の成功記録とは区別する。
- `P105_CAPTURE_VERDICT.json` はケース別判定、停止・再開イベント、追加カメラの実撮影・追従座標を含む。`P105_CAMERA_VIEWS.zip` は同じ物理実行の動画3本と監査記録を収める。
- 10/10は**あらかじめ設定した既知の実験ケースでの成功率**である。未知果実への汎化性能、実機ロボットの安全認証、全ケースの1個3秒以内、傷の全周自動検出は検証対象外である。
- エラー時には `/content/P107_CAPTURE_ONCE.log` と新規の作業フォルダを確認し、上書き・自動再実行によって成功済みデータを置き換えない。
- GitHubには配布ZIPをそのまま1個だけ置かず、**展開したファイル一式**をPublicリポジトリへ登録する。追加した背景画像は `.gitignore` の例外設定で公開対象に含める。

### 物理実行と提出用動画編集の区別

このノートブックは**物理試行・実録撮影・監査まで**を実行する。提出用の背景合成は `submission_video/p105_compose.py` で別途実行する。完成予想図を実機映像と誤認させず、実際のPyBullet映像は実録として明示する。
